# Prepare Khmer comparison dataset
Download, clean and sample 519 standalone Khmer questions from metythorn/khmer-answer-question. Source: https://huggingface.co/datasets/metythorn/khmer-answer-question (Apache-2.0).

In [ ]:
%pip install -q datasets

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from datasets import load_dataset
import pandas as pd
import re, unicodedata
from pathlib import Path

folder = Path('/content/drive/My Drive/ColabNotebooks/I5/ADV')
df = load_dataset('metythorn/khmer-answer-question', split='train').to_pandas()

def clean(x):
    if pd.isna(x): return ''
    x = unicodedata.normalize('NFC', str(x)).replace('\u200b', '').replace('\ufeff', '').strip()
    return '' if x.lower() in {'nan', 'none', 'null'} else x

for col in ['instruction', 'input', 'output']:
    df[col] = df[col].map(clean)
# Select standalone questions to match the banking FAQ task.
df = df[df['input'].eq('') & df['instruction'].ne('') & df['output'].ne('')].copy()
df = df[df['instruction'].str.contains(r'[?？]$')]
df = df[df['instruction'].str.contains(r'[\u1780-\u17ff]') & df['output'].str.contains(r'[\u1780-\u17ff]')]
df = df[['instruction', 'output']].rename(columns={'instruction':'input'}).drop_duplicates(subset=['input'])
assert len(df) >= 519, f'Only {len(df)} eligible questions'
result = df.sample(n=519, random_state=3407).reset_index(drop=True)
path = folder / 'data_kh_general.csv'
result.to_csv(path, index=False, encoding='utf-8-sig')
check = pd.read_csv(path)
assert check.shape == (519, 2) and not check.isna().any().any()
assert not check['input'].duplicated().any()
print('PASS:', path, check.shape)
display(check.head())

README.md:   0%|          | 0.00/6.78k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 31.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/41601 [00:00<?, ? examples/s]

PASS: /content/drive/My Drive/ColabNotebooks/I5/ADV/data_kh_general.csv (519, 2)


,input,output
0,តើអ្នកនឹងពន្យល់ពីកម្មវិធីកុំព្យូទ័រដល់ក្មេងអាយ...,ការសរសេរកម្មវិធីកុំព្យូទ័រគឺដូចជាការផ្តល់ការណែ...
1,ឆ្លើយសំណួរ៖ តើបណ្តាញសរសៃប្រសាទសិប្បនិម្មិតជាអ្វី?,បណ្តាញសរសៃប្រសាទសិប្បនិមិត្ត (ANN) គឺជាប្រភេទន...
2,តើចិតសិបប្រាំលានក្នុងទម្រង់ជាលេខប៉ុន្មាន?,"ចិតសិបប្រាំលានក្នុងទម្រង់ជាលេខគឺ 75,000,000។"
3,តើថ្ងៃទី១៥ ខែមិថុនា មានអត្ថន័យយ៉ាងណា?,ថ្ងៃទី 15 ខែមិថុនា គឺជាកាលបរិច្ឆេទដែលមានព្រឹត្...
4,តើអ្នកអាចធ្វើយ៉ាងណាក្នុងការកាត់បន្ថយបរិមាណប្លា...,មានវិធីជាច្រើនដែលអ្នកអាចកាត់បន្ថយបរិមាណផ្លាស្ទ...
